# Viora — Free GPU Training (Colab / Kaggle)

Train Viora's **own** video-language model for **$0** on a free **T4 (16 GB)** — plenty for the
SigLIP + Qwen-0.5B + LoRA ("pragmatic") model. Free sessions time out (4–12 h), so this notebook
**checkpoints** to Drive/output and can **resume** across sessions.

**Before running:** enable the GPU.
- **Colab:** Runtime → Change runtime type → **T4 GPU**
- **Kaggle:** Settings → Accelerator → **GPU T4 x2** (30 free GPU-hrs/week)

This is Viora's own model — no wrapper, no external answering API.

In [ ]:
# 1) Confirm the GPU + torch actually work together (catches a broken/mismatched torch early).
import torch
print("torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU! Set Accelerator to GPU T4 (Kaggle Settings / Colab Runtime)."
try:
    _ = (torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")).sum().item()
    print("CUDA kernel test: OK ✓  (torch matches this GPU)")
except Exception as e:
    raise SystemExit(
        f"torch cannot run on this GPU: {e}\n\n"
        ">> FIX: FACTORY RESET the session to restore the correct torch:\n"
        "     Kaggle: top-right '...' menu -> Factory reset   |   Colab: Runtime -> Disconnect and delete runtime\n"
        ">> Then re-run. Do NOT let anything pip-install/upgrade torch (cell 3 uses --no-deps to avoid that)."
    )

In [ ]:
# 2) Get the Viora code from GitHub (absolute paths; re-run pulls the latest fixes)
import os
os.environ["GIT_TERMINAL_PROMPT"] = "0"  # fail fast instead of hanging on a login prompt

REPO_URL = "https://github.com/garvbahl37-gif/Viora.git"
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(BASE, "viora")

# If the repo is PRIVATE: add a GitHub token in Kaggle (Add-ons -> Secrets) as
# GITHUB_TOKEN, then uncomment the next two lines:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@github.com/garvbahl37-gif/Viora.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    rc = os.system(f"cd {REPO_DIR} && git pull --ff-only")   # already cloned -> update to latest
else:
    rc = os.system(f"git clone {REPO_URL} {REPO_DIR}")
if rc != 0:
    raise SystemExit(
        "Git failed. Fix one of these:\n"
        "  1) Make the repo PUBLIC (simplest), or set GITHUB_TOKEN above for a private repo.\n"
        "  2) Enable Internet: Kaggle right panel -> Internet -> On (phone-verify your account)."
    )
%cd {REPO_DIR}
print("working dir:", os.getcwd())

In [ ]:
# 3) Install Viora WITHOUT reinstalling torch.
#    Kaggle/Colab ship a CUDA-matched torch; letting pip pull torch from PyPI causes
#    "CUDA error: no kernel image is available for execution on the device".
#    So: install the package with --no-deps, then install only the extra deps explicitly.
import torch
print("keeping torch", torch.__version__, "| cuda", torch.cuda.is_available())
!pip install -q -e . --no-deps
!pip install -q einops omegaconf pyyaml tqdm rich av webdataset peft \
    transformers safetensors huggingface_hub
# Kaggle/Colab ship an old torchao (0.10) that breaks peft's LoRA dispatch (also handled in code).
!pip uninstall -q -y torchao 2>/dev/null || true
!python scripts/validate_environment.py

In [ ]:
# 4) Choose an output dir that SURVIVES session end (so you can resume).
#    Colab -> Google Drive;  Kaggle -> /kaggle/working (downloadable).
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/viora_runs/pragmatic'
except Exception:
    OUT = '/kaggle/working/pragmatic' if os.path.isdir('/kaggle') else 'runs/pragmatic'
os.makedirs(OUT, exist_ok=True)
print('checkpoints ->', OUT)

In [ ]:
# 5) Data. Synthetic shards so the whole pipeline runs end-to-end for free.
#    Swap in a real dataset (MSR-VTT) later for a useful model — see docs/PRODUCTION.md.
!python scripts/build_shards.py --synthetic 3000 --out data/shards/train-%06d.tar
import glob
_n = len(glob.glob("data/shards/train-*.tar"))          # robust to the actual shard count
SHARDS = "data/shards/train-{000000..%06d}.tar" % (_n - 1)
print(f"{_n} shards -> {SHARDS}")

In [ ]:
# 6) Train: LoRA on Qwen-0.5B + frozen SigLIP + Viora's trainable bridge.
#    T4 supports fp16 (NOT bf16); small num_workers for Kaggle's limited CPUs.
#    Checkpoints to {OUT} every 200 steps -> resumable across free sessions.
#    (If you hit out-of-memory, lower training.batch_size to 2.)
!python scripts/train.py \
  --model configs/model/viora_pragmatic.yaml \
  --train configs/training/pragmatic_lora.yaml \
  --shards "{SHARDS}" \
  llm.name_or_path=Qwen/Qwen2.5-0.5B-Instruct \
  training.precision=fp16 training.batch_size=4 training.num_workers=2 \
  training.gradient_checkpointing=true \
  training.max_steps=3000 training.save_every=200 training.log_every=20 \
  training.output_dir={OUT}

## Resuming after a session times out

Re-run cells 1–5, then add `training.resume=<checkpoint>` to cell 6 — e.g.:

```
  training.resume={OUT}/step_2000.pt
```

It restores model + optimizer + step and continues. Repeat across free sessions until done.

## Get real answers

Synthetic data proves the loop; for a *useful* model, upload a real video-QA set (MSR-VTT ~7 GB)
as a Kaggle Dataset / to Drive, convert with `scripts/build_shards.py`, and point `--shards` at it.
Bigger LLM (`Qwen/Qwen2.5-1.5B-Instruct`) + more data + more hours = better quality.

## Serve it

```
VIORA_MODEL_CONFIG=configs/model/viora_pragmatic.yaml VIORA_CHECKPOINT={OUT}/final.pt \
  uvicorn viora.serving.api:app --host 0.0.0.0 --port 8000
```